# Gemma 4 E4B Legal AI — GRPO Fine-tuning with Unsloth

**Model**: `unsloth/gemma-4-e4b-it-bnb-4bit` (4B active params, multimodal: text + vision + audio)

**Training Method**: GRPO (Group Relative Policy Optimization) — RL-based alignment

**Hardware**: Colab A100 (40GB) recommended, T4 (15GB) feasible with reduced batch

**Datasets**:
- 60K legal documents (HuggingFace — auto-download)
- 200-500 codebase patterns (local upload)
- Legal reward functions for GRPO (citation accuracy, statute references, reasoning chain)

**Target**: RTX 3060 Ti deployment via Ollama GGUF Q4_K_M (~3.5GB VRAM)

**Why Gemma 4 E4B?**:
- Apache 2.0 license (fully open)
- Text + Vision + Audio in ONE model (replaces separate VLM pipeline)
- ~6GB Q4 fits RTX 3060 Ti 8GB with room for embeddings
- Day-0 TRT-LLM support from NVIDIA

**LoRA Adapter Note**: Existing Gemma 3 adapters (5-7hr training) are NOT compatible.
Different architecture = different weight dimensions. Training data (.jsonl) IS reusable.

---

## Prerequisites

1. Extract local datasets:
   ```bash
   cd sveltekit-frontend
   bash ../scripts/dataset-collection/extract-legal-patterns.sh
   ```
2. Verify output: `ls training-datasets/` (7 .jsonl files)
3. Runtime → Change runtime type → **A100 GPU** (recommended) or T4

**Training time**: ~3-5 hours (GRPO is slower than SFT due to multiple rollouts)
**Output size**: ~3 GB (Q4_K_M GGUF)
**Cost**: ~$8-12 (Colab Pro+ A100)

## 1. Setup

In [ ]:
# Install Unsloth (latest from GitHub — GRPO support required)
!pip uninstall unsloth -y
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install bitsandbytes accelerate peft trl transformers datasets huggingface_hub pillow

In [ ]:
import torch
import json
import re
from pathlib import Path
from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import load_dataset, concatenate_datasets, Dataset

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram_gb:.1f} GB")
    if vram_gb < 15:
        print(f"\nWARNING: {vram_gb:.1f}GB may be tight for GRPO (multiple rollouts).")
        print("  Reduce num_generations to 2 and batch_size to 1.")
    elif vram_gb >= 40:
        print("\nA100 detected — full GRPO config available.")

## 2. Model Configuration

In [ ]:
# Gemma 4 E4B — 4B active params, efficient architecture
# Check Unsloth's model collection for the exact slug:
#   https://huggingface.co/collections/unsloth/gemma-4
# Common patterns: unsloth/gemma-4-e4b-it-bnb-4bit or unsloth/gemma-4-4b-it-bnb-4bit
MODEL_NAME = "unsloth/gemma-4-e4b-it-bnb-4bit"
MAX_SEQ_LENGTH = 4096  # Gemma 4 supports long context

# LoRA configuration
LORA_R = 16          # Rank 16 for E4B (smaller model = can afford higher rank)
LORA_ALPHA = 16
LORA_DROPOUT = 0     # 0 dropout for GRPO (Unsloth recommendation)

# GRPO configuration
if torch.cuda.is_available():
    _vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    NUM_GENERATIONS = 4 if _vram >= 40 else 2  # Rollouts per prompt
else:
    NUM_GENERATIONS = 2

print(f"Model: {MODEL_NAME}")
print(f"Max seq length: {MAX_SEQ_LENGTH}")
print(f"LoRA rank: {LORA_R}")
print(f"GRPO generations per prompt: {NUM_GENERATIONS}")

## 3. Load Model + Add LoRA

In [ ]:
print(f"Loading {MODEL_NAME}...\n")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,  # Auto-detect (bfloat16 if supported)
)

print(f"Loaded: {MODEL_NAME}")
print(f"BFloat16: {is_bfloat16_supported()}")

In [ ]:
print("Adding LoRA adapters...\n")

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention
        "gate_proj", "up_proj", "down_proj",       # MLP
    ],
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=True,  # Rank-stabilized LoRA (prevents collapse)
)

model.print_trainable_parameters()

## 4. Load Legal Datasets

In [ ]:
def standardize_text(example):
    if 'text' not in example:
        return example
    if isinstance(example['text'], list):
        example['text'] = ' '.join([
            item['value'] if isinstance(item, dict) and 'value' in item else str(item)
            for item in example['text']
        ])
    elif not isinstance(example['text'], str):
        example['text'] = str(example['text'])
    return example

print("Loading HuggingFace legal datasets...\n")

# 1. FineTome
print("[1/7] FineTome...")
dataset1 = load_dataset("mlabonne/FineTome-100k", split="train[:10000]")
dataset1 = dataset1.rename_column('conversations', 'text')
print(f"  {len(dataset1):,}")

# 2. GSM8K
print("[2/7] GSM8K...")
dataset2 = load_dataset("openai/gsm8k", "main", split="train[:5000]")
dataset2 = dataset2.rename_column('question', 'text')
print(f"  {len(dataset2):,}")

# 3. Pile of Law
print("[3/7] Pile of Law...")
pile_of_law = load_dataset("lamblamb/pile_of_law_subset", split="train[:20000]")
print(f"  {len(pile_of_law):,}")

# 4. LEDGAR
print("[4/7] LEDGAR...")
ledgar = load_dataset("lex_glue", "ledgar", split="train[:10000]")
print(f"  {len(ledgar):,}")

# 5. MultiLexSum
print("[5/7] MultiLexSum...")
multilexsum = load_dataset("allenai/multi_lexsum", name="v20230518", split="train[:5000]")
multilexsum = multilexsum.rename_column('summary/short', 'text')
print(f"  {len(multilexsum):,}")

# 6. Case Hold
print("[6/7] Case Hold...")
case_hold = load_dataset("lighteval/lexglue", name="case_hold", split="train[:5000]")
case_hold = case_hold.rename_column('input', 'text')
print(f"  {len(case_hold):,}")

# 7. SCOTUS
print("[7/7] SCOTUS...")
scotus = load_dataset("lighteval/lexglue", name="scotus", split="train[:5000]")
scotus = scotus.rename_column('input', 'text')
print(f"  {len(scotus):,}")

# Standardize
print("\nStandardizing...")
legal_datasets = []
for ds in [dataset1, dataset2, pile_of_law, ledgar, multilexsum, case_hold, scotus]:
    ds = ds.select_columns(['text']).map(standardize_text, num_proc=4)
    legal_datasets.append(ds)

legal_dataset = concatenate_datasets(legal_datasets)
print(f"\nLegal datasets: {len(legal_dataset):,} examples")

## 5. Upload Codebase Datasets

**Before running**: Extract training data locally:
```bash
cd sveltekit-frontend
bash ../scripts/dataset-collection/extract-legal-patterns.sh
```

Upload all 7 .jsonl files from `training-datasets/`

In [ ]:
from google.colab import files

print("Upload your training-datasets/*.jsonl files")
print("(Select all 7 files at once)\n")

uploaded = files.upload()

codebase_patterns = []
for filename, content in uploaded.items():
    if filename.endswith('.jsonl'):
        print(f"Loading {filename}...")
        lines = content.decode('utf-8').strip().split('\n')
        for line in lines:
            if line.strip():
                try:
                    codebase_patterns.append(json.loads(line))
                except json.JSONDecodeError:
                    continue

print(f"\nCodebase patterns: {len(codebase_patterns):,} examples")

## 6. Prepare GRPO Prompt Dataset

GRPO needs a dataset of **prompts** (not prompt-response pairs).
The model generates multiple completions per prompt, then a reward function scores them.

In [ ]:
# Build GRPO prompt dataset from legal texts
# Each entry is a prompt that the model will complete during training

grpo_prompts = []

# Category 1: Legal analysis prompts (from legal dataset text snippets)
for i, example in enumerate(legal_dataset):
    text = example.get('text', '')
    if len(text) < 50:
        continue
    
    # Extract first 200 chars as context, ask for analysis
    snippet = text[:200].strip()
    
    if any(kw in text.lower() for kw in ['statute', 'u.s.c', 'code', 'section']):
        prompt = f"Analyze the following legal statute and explain its implications:\n\n{snippet}..."
    elif any(kw in text.lower() for kw in ['court', 'judge', 'ruling', 'opinion']):
        prompt = f"Summarize the key holdings and reasoning in this court opinion:\n\n{snippet}..."
    elif any(kw in text.lower() for kw in ['contract', 'agreement', 'party', 'clause']):
        prompt = f"Review this contract provision and identify key legal terms and obligations:\n\n{snippet}..."
    elif any(kw in text.lower() for kw in ['evidence', 'testimony', 'witness']):
        prompt = f"Analyze this evidence description for a legal investigation:\n\n{snippet}..."
    else:
        prompt = f"Provide legal analysis of the following:\n\n{snippet}..."
    
    grpo_prompts.append({
        "prompt": prompt,
        "reference_text": text[:500],  # For reward function comparison
    })
    
    if len(grpo_prompts) >= 10000:  # Cap at 10K (reduced from 15K for <5hr training)
        break

# Category 2: Codebase pattern prompts
for pattern in codebase_patterns:
    text = pattern.get('text', '')
    if len(text) < 30:
        continue
    
    if any(kw in text.lower() for kw in ['$state', 'svelte', 'runes']):
        prompt = f"Explain this Svelte 5 pattern and its correct usage:\n\n{text[:150]}..."
    elif any(kw in text.lower() for kw in ['evidence', 'pipeline', 'rag']):
        prompt = f"Describe this legal AI evidence processing pattern:\n\n{text[:150]}..."
    elif any(kw in text.lower() for kw in ['tensorrt', 'triton', 'gpu']):
        prompt = f"Explain this AI inference deployment concept:\n\n{text[:150]}..."
    else:
        prompt = f"Explain the following technical concept:\n\n{text[:150]}..."
    
    grpo_prompts.append({
        "prompt": prompt,
        "reference_text": text[:500],
    })

# Category 3: Freeform legal reasoning prompts (no reference — tests generalization)
freeform_prompts = [
    "What are the elements of a negligence claim under common law?",
    "Explain the difference between civil and criminal burden of proof.",
    "Summarize the key provisions of 42 U.S.C. Section 1983.",
    "What is the chain of custody requirement for physical evidence?",
    "Describe the hearsay rule and its major exceptions under the Federal Rules of Evidence.",
    "What factors do courts consider when deciding motions for summary judgment?",
    "Explain how Daubert v. Merrell Dow applies to expert witness testimony.",
    "What are the Miranda rights and when must they be given?",
    "Describe the differences between express and implied contracts.",
    "What constitutes a valid search warrant under the Fourth Amendment?",
    "Explain the doctrine of res judicata and collateral estoppel.",
    "What are the remedies available in a breach of contract action?",
    "Describe the standard for granting preliminary injunctive relief.",
    "What is the attorney-client privilege and when can it be waived?",
    "Explain the concept of joint and several liability in tort law.",
]

for p in freeform_prompts:
    grpo_prompts.append({"prompt": p, "reference_text": ""})

# Category 4: Tool calling + web search prompts (trains agentic behavior)
tool_calling_prompts = [
    # glossary_search tool
    'I need the legal definition of "habeas corpus". Use the glossary_search tool to look it up.',
    "Define the term 'stare decisis' using the glossary_search tool.",
    "What does 'voir dire' mean? Search the legal glossary.",
    "Look up the definition of 'amicus curiae' in the legal glossary.",
    # rag_search tool
    "Search our evidence database for documents related to the Fourth Amendment violation claims.",
    "Use rag_search to find evidence about chain of custody failures in this case.",
    "Find relevant legal documents about qualified immunity in our knowledge base.",
    "Search the document database for precedents involving police body camera footage.",
    # web_search tool
    "Search the web for the latest Supreme Court rulings on qualified immunity from 2024-2025.",
    "Use web_search to find recent case law on digital evidence admissibility.",
    "Look up the current federal sentencing guidelines for wire fraud online.",
    "Search for recent amendments to the Federal Rules of Civil Procedure.",
    # graph_expand tool
    "Show me the knowledge graph connections for this case — what evidence links to what?",
    "Expand the graph to find related documents connected to evidence item E-001.",
    "What are the graph neighbors for this case? Show related cases and evidence connections.",
    # authority_drill tool
    "Drill down into 42 U.S.C. § 1983 — find the full text and related authorities.",
    "Trace the authority chain for Monell v. Department of Social Services.",
    "What statutes and cases cite Graham v. Connor? Drill into the precedent chain.",
    # case_search tool
    "Find similar cases to this one involving excessive force by law enforcement.",
    "Search for cases with similar fact patterns involving digital evidence spoliation.",
    "Are there precedents similar to our case? Search for matching cases by legal issue.",
    # Multi-tool scenarios
    "First search our documents for qualified immunity precedents, then search the web for any recent updates.",
    "Look up 'proximate cause' in the glossary, then find relevant case documents about causation.",
    "Search for related evidence in the graph, then drill down into the statutes they reference.",
]

for p in tool_calling_prompts:
    grpo_prompts.append({"prompt": p, "reference_text": ""})

grpo_dataset = Dataset.from_list(grpo_prompts)
print(f"GRPO prompt dataset: {len(grpo_dataset):,} prompts")
print(f"  Legal analysis: ~{min(10000, len(legal_dataset)):,}")
print(f"  Codebase patterns: {len(codebase_patterns):,}")
print(f"  Freeform reasoning: {len(freeform_prompts)}")
print(f"  Tool calling/web search: {len(tool_calling_prompts)}")
print(f"\nExample: {grpo_dataset[0]['prompt'][:100]}...")

## 7. Define Legal Reward Functions

GRPO uses reward functions to score model completions.
We use a multi-signal rubric (7 functions):
- **Citation accuracy** (0.20) — valid legal citation formats
- **Reasoning chain** (0.20) — logical structure (because/therefore/thus)
- **Legal formatting** (0.15) — section headers, numbered lists, proper terms
- **Web search grounding** (0.15) — source attribution, hedging, URL refs
- **Tool calling format** (0.10) — proper JSON tool call structure
- **Hallucination penalty** (0.10) — penalize fabricated case names
- **Length quality** (0.10) — not too short, not too verbose

In [ ]:
import re

# Legal citation patterns (Bluebook format)
CITATION_RE = re.compile(
    r'\d+\s+[A-Z][a-z]*\.?\s*(?:2d|3d|4th|Supp\.?)?\s*\d+',  # e.g., "347 U.S. 483"
    re.IGNORECASE
)
STATUTE_RE = re.compile(
    r'\d+\s+U\.?S\.?C\.?\s*(?:§|Section)?\s*\d+',  # e.g., "42 U.S.C. § 1983"
    re.IGNORECASE
)
CFR_RE = re.compile(
    r'\d+\s+C\.?F\.?R\.?\s*(?:§|Part)?\s*\d+',  # e.g., "29 CFR Part 1910"
    re.IGNORECASE
)
REASONING_WORDS = [
    'because', 'therefore', 'thus', 'consequently', 'accordingly',
    'moreover', 'furthermore', 'however', 'whereas', 'given that',
    'in light of', 'pursuant to', 'under', 'based on', 'considering',
]
LEGAL_TERMS = [
    'plaintiff', 'defendant', 'court', 'ruling', 'statute',
    'precedent', 'jurisdiction', 'liability', 'negligence', 'breach',
    'evidence', 'testimony', 'motion', 'injunction', 'damages',
    'amendment', 'constitutional', 'due process', 'burden of proof',
]
HALLUCINATED_CASE_RE = re.compile(
    r'(?:Smith|Jones|Doe)\s+v\.\s+(?:Smith|Jones|Doe|United States|State)',
    re.IGNORECASE
)


def reward_citation_accuracy(completions, **kwargs):
    """Reward valid legal citations (Bluebook format)"""
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else str(completion)
        citations = CITATION_RE.findall(text)
        statutes = STATUTE_RE.findall(text)
        cfr = CFR_RE.findall(text)
        total = len(citations) + len(statutes) + len(cfr)
        score = min(1.0, total * 0.5)
        rewards.append(score)
    return rewards


def reward_reasoning_chain(completions, **kwargs):
    """Reward logical reasoning structure"""
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else str(completion)
        text_lower = text.lower()
        reasoning_count = sum(1 for word in REASONING_WORDS if word in text_lower)
        score = min(1.0, reasoning_count * 0.25)
        rewards.append(score)
    return rewards


def reward_legal_formatting(completions, **kwargs):
    """Reward professional legal formatting"""
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else str(completion)
        text_lower = text.lower()
        score = 0.0
        term_count = sum(1 for term in LEGAL_TERMS if term in text_lower)
        score += min(0.4, term_count * 0.1)
        if re.search(r'\d+\.\s', text):
            score += 0.2
        if re.search(r'(?:First|Second|Third|Finally|In conclusion)', text):
            score += 0.2
        if text.count('\n') >= 2:
            score += 0.2
        rewards.append(min(1.0, score))
    return rewards


def reward_web_search_grounding(completions, **kwargs):
    """Reward proper source attribution and web search integration"""
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else str(completion)
        score = 0.0
        # Source attribution phrases
        if re.search(r'(?:according to|source:|per |as stated in|as reported)', text, re.I):
            score += 0.25
        # URL/domain references (legal sites)
        if re.search(r'(?:https?://|\.gov|\.edu|\.org|law\.cornell|westlaw|lexisnexis)', text, re.I):
            score += 0.25
        # Temporal hedging (good for web-grounded answers)
        if re.search(r'(?:as of|reportedly|appears to|based on available|current as of)', text, re.I):
            score += 0.25
        # Structured source citations [1] or (Source N)
        if re.search(r'\[\d+\]|\(Source|\[Source', text):
            score += 0.25
        rewards.append(min(1.0, score))
    return rewards


def reward_tool_calling_format(completions, **kwargs):
    """Reward proper tool calling JSON structure (trains tool-use behavior)"""
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else str(completion)
        score = 0.0
        # Check for tool call JSON structure
        if re.search(r'"name"\s*:\s*"[\w_]+"', text):
            score += 0.3
        if re.search(r'"arguments"\s*:\s*\{', text):
            score += 0.3
        # Check for function-like invocation patterns
        if re.search(r'(?:web_search|search_legal|get_statute|lookup_case)', text, re.I):
            score += 0.2
        # Proper query parameter
        if re.search(r'"query"\s*:\s*"[^"]{5,}"', text):
            score += 0.2
        rewards.append(min(1.0, score))
    return rewards


def reward_anti_hallucination(completions, **kwargs):
    """Penalize obviously hallucinated content"""
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else str(completion)
        score = 1.0
        hallucinated = HALLUCINATED_CASE_RE.findall(text)
        score -= len(hallucinated) * 0.3
        if any(phrase in text.lower() for phrase in ['i cannot', 'i\'m not sure', 'i don\'t have']):
            score -= 0.2
        rewards.append(max(0.0, score))
    return rewards


def reward_length_quality(completions, **kwargs):
    """Reward appropriate response length"""
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else str(completion)
        word_count = len(text.split())
        if word_count < 20:
            score = 0.1
        elif word_count < 50:
            score = 0.5
        elif word_count <= 300:
            score = 1.0
        elif word_count <= 500:
            score = 0.7
        else:
            score = 0.3
        rewards.append(score)
    return rewards


# Verify all 7 reward functions
test_completion = [[{"content": "Under 42 U.S.C. Section 1983, the plaintiff must demonstrate that the defendant, acting under color of state law, deprived them of a right secured by the Constitution. Therefore, the court must first establish jurisdiction. According to law.cornell.edu, as of 2025: 1. Federal question jurisdiction exists. 2. The statute of limitations applies. [Source 1]"}]]

print("Reward function test (7 functions):")
print(f"  Citation accuracy:     {reward_citation_accuracy(test_completion)[0]:.2f}")
print(f"  Reasoning chain:       {reward_reasoning_chain(test_completion)[0]:.2f}")
print(f"  Legal formatting:      {reward_legal_formatting(test_completion)[0]:.2f}")
print(f"  Web search grounding:  {reward_web_search_grounding(test_completion)[0]:.2f}")
print(f"  Tool calling format:   {reward_tool_calling_format(test_completion)[0]:.2f}")
print(f"  Anti-hallucination:    {reward_anti_hallucination(test_completion)[0]:.2f}")
print(f"  Length quality:         {reward_length_quality(test_completion)[0]:.2f}")

## 8. Configure GRPO Training

In [ ]:
from trl import GRPOConfig, GRPOTrainer

# GRPO training configuration — optimized for <5 hour training on A100
grpo_config = GRPOConfig(
    output_dir="./gemma4-e4b-legal-grpo-output",
    
    # GRPO-specific
    num_generations=NUM_GENERATIONS,   # Rollouts per prompt (4 on A100, 2 on T4)
    max_completion_length=384,         # Reduced from 512 — legal responses rarely need more
    max_prompt_length=256,             # Truncate long prompts (saves memory + time)
    
    # Training hyperparams — 1 epoch is sufficient with GRPO reward signal
    num_train_epochs=1,               # Reduced from 2 — GRPO converges faster than SFT
    per_device_train_batch_size=2,    # Bumped from 1 — A100 handles it, T4 auto-adjusts
    gradient_accumulation_steps=4,    # Effective batch = 8 (2*4)
    learning_rate=5e-6,               # Lower LR for RL stability
    
    # Precision
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    
    # Checkpointing — less frequent saves = faster
    logging_steps=10,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    warmup_steps=30,
    
    # Optimizer
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    max_grad_norm=0.5,  # Lower for RL stability
    
    # Speed optimizations
    dataloader_num_workers=4,         # Parallel data loading
    dataloader_pin_memory=True,       # Faster GPU transfer
    
    # Misc
    seed=42,
    report_to="none",
)

print("GRPO Training Config (speed-optimized):")
print(f"  Generations/prompt: {grpo_config.num_generations}")
print(f"  Max completion: {grpo_config.max_completion_length} tokens")
print(f"  Max prompt: {grpo_config.max_prompt_length} tokens")
print(f"  Epochs: {grpo_config.num_train_epochs}")
print(f"  Batch size: {grpo_config.per_device_train_batch_size}")
print(f"  Grad accum: {grpo_config.gradient_accumulation_steps}")
print(f"  Effective batch: {grpo_config.per_device_train_batch_size * grpo_config.gradient_accumulation_steps}")
print(f"  LR: {grpo_config.learning_rate}")
print(f"  Precision: {'bfloat16' if is_bfloat16_supported() else 'float16'}")
print(f"  Estimated: ~2.5-4 hours on A100, ~4-6 hours on T4")

## 9. Initialize GRPO Trainer

In [ ]:
trainer = GRPOTrainer(
    model=model,
    config=grpo_config,
    reward_funcs=[
        reward_citation_accuracy,    # Legal citation format (Bluebook, USC, CFR)
        reward_reasoning_chain,      # Logical connectors (therefore, because, thus)
        reward_legal_formatting,     # Structure (lists, headers, legal terms)
        reward_web_search_grounding, # Source attribution, URL refs, hedging
        reward_tool_calling_format,  # JSON tool call structure
        reward_anti_hallucination,   # Penalize fabricated cases
        reward_length_quality,       # Sweet spot 50-300 words
    ],
    train_dataset=grpo_dataset,
    tokenizer=tokenizer,
)

print("GRPO Trainer initialized")
print(f"  Reward functions: 7 (citation, reasoning, formatting, web-search, tool-calling, anti-hallucination, length)")
print(f"  Training prompts: {len(grpo_dataset):,}")
print(f"  Dataset cap: 10K legal + codebase + freeform + tool-calling")

## 10. Train (3-5 hours)

In [ ]:
print("=" * 70)
print("GRPO TRAINING START")
print("=" * 70)
print(f"Model: Gemma 4 E4B")
print(f"Method: GRPO ({NUM_GENERATIONS} generations/prompt)")
print(f"Prompts: {len(grpo_dataset):,}")
print(f"Epochs: {grpo_config.num_train_epochs}")
print(f"Estimated time: 3-5 hours\n")

trainer_stats = trainer.train()

print("\n" + "=" * 70)
print("GRPO TRAINING COMPLETE")
print("=" * 70)
runtime = trainer_stats.metrics['train_runtime']
print(f"Time: {runtime:.0f}s ({runtime/3600:.1f} hours)")
print(f"Samples/sec: {trainer_stats.metrics['train_samples_per_second']:.2f}")

## 11. Test Inference

In [ ]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

test_prompts = [
    "Explain the elements of a negligence claim and cite relevant case law.",
    "What is the chain of custody requirement for digital evidence?",
    "Describe the RAG evidence upload pipeline in a legal AI system.",
    "Analyze 42 U.S.C. Section 1983 and its application to police misconduct cases.",
]

text_streamer = TextStreamer(tokenizer, skip_prompt=True)

for prompt in test_prompts:
    print("\n" + "=" * 70)
    print(f"Prompt: {prompt}")
    print("=" * 70)
    
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    
    model.generate(
        input_ids=inputs,
        streamer=text_streamer,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
    )
    print()

## 12. Save LoRA Adapters

In [ ]:
model.save_pretrained("gemma4-e4b-legal-grpo-lora")
tokenizer.save_pretrained("gemma4-e4b-legal-grpo-lora")

print("LoRA adapters saved: gemma4-e4b-legal-grpo-lora/")
print("Size: ~200-500 MB (adapters only)")

## 13. Export to GGUF for Ollama

Direct GGUF export via Unsloth — creates Ollama-ready model file.

In [ ]:
# Method 1: Direct GGUF export (preferred — handles merge + quantize in one step)
print("Exporting to GGUF Q4_K_M (Ollama-ready)...")
model.save_pretrained_gguf(
    "gemma4-e4b-legal",
    tokenizer,
    quantization_method="q4_k_m",  # Best quality/size ratio for 8GB GPU
)
print("GGUF saved: gemma4-e4b-legal/")
print("Expected size: ~3 GB (Q4_K_M)")
print()

# Method 2: Also save 16-bit merged for TensorRT conversion
print("Exporting 16-bit merged model (for TRT-LLM)...")
model.save_pretrained_merged(
    "gemma4-e4b-legal-merged-16bit",
    tokenizer,
    save_method="merged_16bit",
)
print("16-bit saved: gemma4-e4b-legal-merged-16bit/")

## 14. Create Ollama Modelfile

In [ ]:
import glob

# Find the GGUF file
gguf_files = glob.glob("gemma4-e4b-legal/*.gguf") + glob.glob("gemma4-e4b-legal/**/*.gguf")
gguf_path = gguf_files[0] if gguf_files else "gemma4-e4b-legal/unsloth.Q4_K_M.gguf"

modelfile_content = f"""FROM {gguf_path}

PARAMETER temperature 0.7
PARAMETER num_predict 2048
PARAMETER num_ctx 8192
PARAMETER top_k 40
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.1

SYSTEM \"\"\"You are a specialized Legal AI Assistant powered by Gemma 4, fine-tuned with GRPO reinforcement learning for legal analysis. You excel at:

- Legal document analysis with proper citation (Bluebook format)
- Evidence classification and chain of custody assessment
- Statute interpretation (U.S.C., CFR, state codes)
- Case law reasoning and precedent analysis
- Contract review and liability assessment
- Legal entity extraction (citations, statutes, dates, monetary amounts)
- Forensic pattern detection in documents

Always provide structured, well-reasoned analysis. Cite specific statutes and case law when relevant. Note that responses are informational analysis, not formal legal advice.\"\"\"

TEMPLATE \"\"\"{{{{ if .System }}}}<start_of_turn>system
{{{{ .System }}}}<end_of_turn>
{{{{ end }}}}{{{{ if .Prompt }}}}<start_of_turn>user
{{{{ .Prompt }}}}<end_of_turn>
<start_of_turn>model
{{{{ end }}}}{{{{ .Response }}}}<end_of_turn>\"\"\"
"""

with open("gemma4-e4b-legal/Modelfile", "w") as f:
    f.write(modelfile_content)

print("Modelfile written: gemma4-e4b-legal/Modelfile")
print()
print("To deploy on your local machine:")
print("  1. Download gemma4-e4b-legal/ directory")
print("  2. cd gemma4-e4b-legal/")
print(f"  3. ollama create gemma4-legal:latest -f Modelfile")
print("  4. ollama run gemma4-legal:latest")
print()
print("OR use existing merge-and-export.sh:")
print("  ./scripts/unsloth-training/merge-and-export.sh \\")
print("    --adapter gemma4-e4b-legal-grpo-lora \\")
print("    --base google/gemma-4-e4b-it \\")
print("    --name gemma4-legal")

## 15. Package for Download

In [ ]:
# Zip GGUF model for download
!zip -r gemma4-e4b-legal-gguf.zip gemma4-e4b-legal/

print("\nPackaged: gemma4-e4b-legal-gguf.zip (~3 GB)")
print("\nDownload and deploy:")
print("  1. Download gemma4-e4b-legal-gguf.zip")
print("  2. unzip gemma4-e4b-legal-gguf.zip")
print("  3. cd gemma4-e4b-legal/")
print("  4. ollama create gemma4-legal:latest -f Modelfile")
print("  5. Update .env: LLM_MODEL=gemma4-legal:latest")
print()
print("RTX 3060 Ti VRAM: ~3.5GB (Q4_K_M) + ~0.6GB (embeddinggemma) = ~4.1GB")
print("Remaining: ~4GB free for TensorRT / custom CUDA ops")

# Optional: Auto-download in Colab
# from google.colab import files
# files.download('gemma4-e4b-legal-gguf.zip')

## 16. (Optional) SFT Warm-up Before GRPO

If GRPO alone isn't producing strong enough results, you can do a
short SFT (Supervised Fine-Tuning) warm-up first, THEN apply GRPO.
This is the "SFT → GRPO" two-stage pipeline used by DeepSeek-R1.

Run this cell BEFORE cells 8-10 if you want the two-stage approach.

In [ ]:
# OPTIONAL: SFT warm-up (run BEFORE GRPO training)
# Uncomment to enable

# from trl import SFTTrainer
# from transformers import TrainingArguments
#
# # Prepare SFT dataset (prompt-response pairs)
# def format_sft(example):
#     text = example.get('text', '')
#     instruction = "Analyze the following legal concept:"
#     return {
#         "conversations": [
#             {"role": "user", "content": instruction},
#             {"role": "assistant", "content": text}
#         ]
#     }
#
# sft_dataset = legal_dataset.map(format_sft, remove_columns=['text'], num_proc=4)
#
# sft_args = TrainingArguments(
#     output_dir="./gemma4-sft-warmup",
#     num_train_epochs=1,  # Just 1 epoch for warm-up
#     per_device_train_batch_size=2,
#     gradient_accumulation_steps=8,
#     learning_rate=2e-4,
#     fp16=not is_bfloat16_supported(),
#     bf16=is_bfloat16_supported(),
#     logging_steps=10,
#     save_strategy="no",  # Don't save SFT checkpoints
#     optim="adamw_8bit",
#     warmup_steps=50,
#     report_to="none",
# )
#
# sft_trainer = SFTTrainer(
#     model=model,
#     tokenizer=tokenizer,
#     train_dataset=sft_dataset,
#     max_seq_length=MAX_SEQ_LENGTH,
#     args=sft_args,
#     dataset_text_field="conversations",
#     packing=False,
# )
#
# print("SFT Warm-up (1 epoch)...")
# sft_trainer.train()
# print("SFT warm-up complete. Now run GRPO cells (8-10).")

---

## Summary

**What we trained**:
- Base: Gemma 4 E4B (4B active params, multimodal: text + vision + audio)
- Method: GRPO reinforcement learning with 7 legal reward functions
- Data: ~10K legal documents + codebase patterns + freeform reasoning + tool calling prompts
- Output: GGUF Q4_K_M (~3GB) for Ollama deployment

**Reward Functions (7)**:
| Function | Weight | Signal |
|----------|--------|--------|
| Citation accuracy | ~0.14 | Bluebook, U.S.C., CFR formats |
| Reasoning chain | ~0.14 | Logical connectors (therefore, because, thus) |
| Legal formatting | ~0.14 | Numbered lists, section headers, legal terms |
| Web search grounding | ~0.14 | Source attribution, URL refs, temporal hedging |
| Tool calling format | ~0.14 | JSON tool call structure (name, arguments, query) |
| Anti-hallucination | ~0.14 | Penalize fabricated case names |
| Length quality | ~0.14 | Sweet spot: 50-300 words |

**Speed Optimizations (vs previous 5+ hour run)**:
- Dataset cap: 15K → 10K prompts
- Epochs: 2 → 1 (GRPO converges faster)
- max_completion_length: 512 → 384
- max_prompt_length: 256 (new — truncates long prompts)
- Batch size: 1 → 2 (A100 headroom)
- Save frequency: 200 → 500 steps
- Dataloader workers: 4 + pin_memory
- **Estimated: ~2.5-4 hours on A100**

**Deployment**:
1. Download `gemma4-e4b-legal-gguf.zip` (~3 GB)
2. `ollama create gemma4-legal:latest -f Modelfile`
3. Update `.env`: `LLM_MODEL=gemma4-legal:latest`
4. Update `ollama.ts`: `VLM_MODELS.legal = 'gemma4-legal:latest'`

**VRAM Budget (RTX 3060 Ti 8GB)**:
| Model | VRAM | Purpose |
|-------|------|---------|
| gemma4-legal (Q4_K_M) | ~3.5GB | Text + Vision LLM |
| embeddinggemma | ~0.6GB | 768-dim embeddings |
| CUDA ops (LibTorch) | ~0.5GB | Custom matrix computations |
| KV cache + overhead | ~1.5GB | Inference working memory |
| **Total** | **~6.1GB** | Fits in 8GB |

**Key Advantage**: Gemma 4 E4B does text + vision in ONE model.
Replaces the current dual-model setup (gemma3-legal for text + separate VLM pipeline).
The `vlm-evidence-analyzer.ts` can use the same `gemma4-legal:latest` model for both
text analysis and image analysis via Ollama's `images: [base64]` parameter.

**Existing merge pipeline works**:
```bash
./scripts/unsloth-training/merge-and-export.sh \
  --adapter gemma4-e4b-legal-grpo-lora \
  --base google/gemma-4-e4b-it \
  --name gemma4-legal
```

**Sources**:
- [Gemma 4 Model Card](https://ai.google.dev/gemma/docs/gemma4)
- [Unsloth GRPO Guide](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide)
- [GRPO Paper (DeepSeek)](https://arxiv.org/abs/2402.03300)
- [Unsloth GGUF Export](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)
- [Unsloth Ollama Export](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-ollama)